# Bug en `crps_from_quantiles` reproducido con codigo de usuario

Este es el bug que **si** he corregido (`skforecast/metrics/metrics.py`). El notebook
mantiene el mismo criterio que las auditorias anteriores: todo se dispara con la API
publica y con flujos que un usuario ejecuta de verdad.

La implementacion anterior elegia los limites de integracion multiplicando los cuantiles
extremos por constantes:

```python
xmin = np.min(pred_quantiles) * 0.9
xmax = np.max(pred_quantiles) * 1.1
```

Eso hace que el resultado dependa del **origen de la escala** de la serie, algo que el CRPS
no debe notar. Consecuencias medidas mas abajo:

| # | Sintoma | Cuando aparece |
|---|---------|----------------|
| 1 | El CRPS cambia al desplazar la serie una constante | Cualquier serie que no este centrada cerca de 0 |
| 2 | El CRPS sale **negativo** | Series de valores negativos (temperaturas bajo cero, saldos, anomalias) |
| 3 | Un valor real muy alejado apenas se penaliza | Outliers, picos de demanda |
| 4 | El CRPS es **0** aunque la prediccion sea horrible | Series intermitentes cuyos cuantiles predichos son todos 0 |
| 5 | Perdida de resolucion numerica | Series con valores grandes (ventas, poblacion, precios) |

In [1]:
import warnings

import numpy as np
import pandas as pd
from scipy.stats import norm
from sklearn.linear_model import LinearRegression

import skforecast
from skforecast.metrics import crps_from_quantiles
from skforecast.recursive import ForecasterRecursive

warnings.simplefilter("ignore")

print("skforecast", skforecast.__version__)
print("numpy     ", np.__version__)
print("pandas    ", pd.__version__)

skforecast 0.24.0
numpy      2.5.2
pandas     2.3.3


---
## La implementacion anterior

La reproduzco tal cual para poder comparar lado a lado. Es literalmente el cuerpo de la
funcion antes de la correccion.

In [2]:
def crps_from_quantiles_antiguo(y_true, pred_quantiles, quantile_levels):
    """Implementacion anterior a la correccion."""

    sorted_indices = np.argsort(pred_quantiles)
    pred_quantiles = pred_quantiles[sorted_indices]
    quantile_levels = quantile_levels[sorted_indices]

    def empirical_cdf(x):
        return np.interp(x, pred_quantiles, quantile_levels, left=0.0, right=1.0)

    def crps_integrand(x):
        return (empirical_cdf(x) - (x >= y_true)) ** 2

    # Integration bounds: Extend slightly beyond predicted quantiles
    xmin = np.min(pred_quantiles) * 0.9
    xmax = np.max(pred_quantiles) * 1.1

    x_values = np.linspace(xmin, xmax, 1000)
    integrand_values = crps_integrand(x_values)

    return float(np.trapezoid(integrand_values, x=x_values))


niveles = np.array([0.1, 0.5, 0.9])
q_doc = np.array([8.0, 10.0, 12.0])
print("ejemplo del docstring (y_true=9.5, cuantiles [8, 10, 12]):")
print("  antiguo   :", crps_from_quantiles_antiguo(9.5, q_doc, niveles))
print("  corregido :", crps_from_quantiles(9.5, q_doc, niveles))

ejemplo del docstring (y_true=9.5, cuantiles [8, 10, 12]):
  antiguo   : 0.46387472397322227
  corregido : 0.4634331094524621


---
## Problema 1. El CRPS depende del origen de la escala

El CRPS es invariante a traslacion: si sumo la misma constante al valor real y a todos los
cuantiles predichos, el error no ha cambiado y la metrica tampoco debe cambiar. Es lo que
pasa al medir la misma temperatura en Celsius o en Kelvin, o al pasar de un indice a su
version reescalada.

Aqui la forma de la prediccion es siempre la misma (cuantiles en `nivel + [-2, 0, 2]`, valor
real en `nivel - 0.5`), solo cambia el nivel de la serie.

In [3]:
pred_rel = np.array([-2.0, 0.0, 2.0])
y_rel = -0.5

filas = []
for nivel in [-20, -10, -5, -1, 0, 1, 5, 10, 20, 100, 1000, 10_000]:
    filas.append(
        {
            "nivel_serie": nivel,
            "antiguo": crps_from_quantiles_antiguo(
                y_rel + nivel, pred_rel + nivel, niveles
            ),
            "corregido": crps_from_quantiles(y_rel + nivel, pred_rel + nivel, niveles),
        }
    )

tabla = pd.DataFrame(filas).set_index("nivel_serie")
print(tabla.round(6).to_string())
print()
print(f"antiguo   min={tabla['antiguo'].min():.6f}  max={tabla['antiguo'].max():.6f}")
print(f"corregido min={tabla['corregido'].min():.6f}  max={tabla['corregido'].max():.6f}")

              antiguo  corregido
nivel_serie                     
-20          0.000000   0.463433
-10          0.372007   0.463433
-5           0.436740   0.463433
-1           0.458145   0.463433
 0           0.460585   0.463433
 1           0.462535   0.463433
 5           0.463513   0.463433
 10          0.463875   0.463433
 20          0.462714   0.463433
 100         0.462785   0.463433
 1000        0.447587   0.463433
 10000       0.366056   0.463433

antiguo   min=0.000000  max=0.463875
corregido min=0.463433  max=0.463433


### Problema 2. Con valores negativos el CRPS sale negativo

Si los cuantiles son negativos, `min * 0.9` queda **por encima** de `max * 1.1` en cuanto el
intervalo es estrecho respecto al nivel de la serie. `np.linspace` genera entonces una
rejilla descendente y `np.trapezoid` devuelve el area con el signo cambiado.

Un CRPS negativo no tiene sentido: la metrica es la integral de un cuadrado y por definicion
es `>= 0`. Ademas rompe cualquier comparacion de modelos, porque "menor es mejor" deja de
significar nada.

In [4]:
# Temperatura minima diaria en invierno: valores negativos y banda estrecha
casos = {
    "temperatura -9.5 C, banda +-0.5": (np.array([-10.0, -9.5, -9.0]), -9.7),
    "temperatura -20 C, banda +-1": (np.array([-21.0, -20.0, -19.0]), -20.4),
    "saldo -1500, banda +-50": (np.array([-1550.0, -1500.0, -1450.0]), -1520.0),
}

for nombre, (q, y) in casos.items():
    a = crps_from_quantiles_antiguo(y, q, niveles)
    b = crps_from_quantiles(y, q, niveles)
    print(nombre)
    print(f"    limites antiguos : xmin={q.min() * 0.9:.4f}  xmax={q.max() * 1.1:.4f}")
    print(f"    antiguo   : {a:>12.6f}   {'<-- NEGATIVO' if a < 0 else ''}")
    print(f"    corregido : {b:>12.6f}")
    print()

temperatura -9.5 C, banda +-0.5
    limites antiguos : xmin=-9.0000  xmax=-9.9000
    antiguo   :    -0.133176   <-- NEGATIVO
    corregido :     0.135397

temperatura -20 C, banda +-1
    limites antiguos : xmin=-18.9000  xmax=-20.9000
    antiguo   :    -0.269382   <-- NEGATIVO
    corregido :     0.270795

saldo -1500, banda +-50
    limites antiguos : xmin=-1395.0000  xmax=-1595.0000
    antiguo   :   -13.542322   <-- NEGATIVO
    corregido :    13.539734



---
## Problema 3. Un valor real fuera del rango predicho apenas se penaliza

Cuando `y_true` cae fuera de los cuantiles predichos, la contribucion correcta al CRPS crece
**linealmente** con la distancia. La version antigua cortaba la integral en `max * 1.1`, asi
que a partir de ahi el error deja de contar.

Es justo el caso que interesa detectar: el pico de demanda que el modelo no vio venir.

In [5]:
q = np.array([8.0, 10.0, 12.0])
print(f"cuantiles predichos: {q}   (limite superior antiguo = {q.max() * 1.1})")
print()
print(f"{'y_true':>8}  {'antiguo':>12}  {'corregido':>12}")
for y in [12.0, 13.0, 15.0, 20.0, 50.0, 100.0]:
    print(
        f"{y:>8}  {crps_from_quantiles_antiguo(y, q, niveles):>12.4f}  "
        f"{crps_from_quantiles(y, q, niveles):>12.4f}"
    )

print()
print("La version antigua se satura: un valor real de 100 se penaliza casi igual que uno")
print("de 15. La corregida crece de forma lineal, que es el comportamiento correcto.")

cuantiles predichos: [ 8. 10. 12.]   (limite superior antiguo = 13.200000000000001)

  y_true       antiguo     corregido
    12.0        1.2148        1.2117
    13.0        2.2118        2.2133
    15.0        2.4130        4.2133
    20.0        2.4130        9.2133
    50.0        2.4130       39.2133
   100.0        2.4130       89.2133

La version antigua se satura: un valor real de 100 se penaliza casi igual que uno
de 15. La corregida crece de forma lineal, que es el comportamiento correcto.


---
## Problema 4. Cuantiles todos a cero: CRPS = 0 con cualquier valor real

En series intermitentes (demanda de repuestos, ventas de productos de baja rotacion) es
habitual que el modelo prediga cero en todos los cuantiles. Con `pred_quantiles` todos a 0,
los limites antiguos quedan `xmin = 0 * 0.9 = 0` y `xmax = 0 * 1.1 = 0`: la rejilla de
integracion colapsa a un punto y el CRPS es exactamente 0.

Es decir, la peor prediccion posible obtiene la mejor puntuacion posible.

In [6]:
q_cero = np.zeros(3)
print("pred_quantiles = [0. 0. 0.]")
print(f"{'venta real':>12}  {'antiguo':>12}  {'corregido':>12}")
for y in [0.0, 1.0, 10.0, 500.0]:
    print(
        f"{y:>12}  {crps_from_quantiles_antiguo(y, q_cero, niveles):>12.4f}  "
        f"{crps_from_quantiles(y, q_cero, niveles):>12.4f}"
    )

print()
print("Prediccion constante distinta de cero (todos los cuantiles iguales a 5):")
q_const = np.full(3, 5.0)
print(f"{'venta real':>12}  {'antiguo':>12}  {'corregido':>12}  {'|y - 5| exacto':>15}")
for y in [5.0, 6.0, 8.0, 20.0]:
    print(
        f"{y:>12}  {crps_from_quantiles_antiguo(y, q_const, niveles):>12.4f}  "
        f"{crps_from_quantiles(y, q_const, niveles):>12.4f}  {abs(y - 5.0):>15.4f}"
    )
print()
print("Para una prediccion determinista el CRPS coincide con el error absoluto.")
print("La version corregida lo cumple, la antigua no.")

pred_quantiles = [0. 0. 0.]
  venta real       antiguo     corregido
         0.0        0.0000        0.0000
         1.0        0.0000        1.0000
        10.0        0.0000       10.0000
       500.0        0.0000      500.0000

Prediccion constante distinta de cero (todos los cuantiles iguales a 5):
  venta real       antiguo     corregido   |y - 5| exacto
         5.0        0.0000        0.0000           0.0000
         6.0        0.5000        1.0000           1.0000
         8.0        0.5000        3.0000           3.0000
        20.0        0.5000       15.0000          15.0000

Para una prediccion determinista el CRPS coincide con el error absoluto.
La version corregida lo cumple, la antigua no.


---
## Caso de usuario: evaluar un forecaster con CRPS

Flujo normal: entrenar un `ForecasterRecursive`, obtener cuantiles con `predict_quantiles` y
resumir la calidad probabilistica con el CRPS medio.

La misma serie se evalua en dos unidades que solo se diferencian en el origen (grados
Celsius y Kelvin). El modelo, las predicciones y los errores son identicos, asi que el CRPS
tiene que ser el mismo.

In [7]:
rng = np.random.default_rng(42)
n = 1000
temp_c = pd.Series(
    5 + 10 * np.sin(2 * np.pi * np.arange(n) / 365) + rng.normal(0, 2.0, n),
    index=pd.date_range("2020-01-01", periods=n, freq="D"),
    name="temperatura",
)
temp_k = temp_c + 273.15

q_levels = np.round(np.arange(0.05, 1.0, 0.05), 2)
TRAIN, TEST = 900, 100


def crps_medio(serie, funcion_crps):
    f = ForecasterRecursive(estimator=LinearRegression(), lags=7)
    f.fit(y=serie.iloc[:TRAIN], store_in_sample_residuals=True)
    pred_q = f.predict_quantiles(
        steps=TEST, quantiles=list(q_levels), n_boot=500, random_state=123
    )
    real = serie.iloc[TRAIN : TRAIN + TEST]
    valores = [
        funcion_crps(float(real.iloc[i]), pred_q.iloc[i].to_numpy(), q_levels)
        for i in range(TEST)
    ]
    return float(np.mean(valores))


print(f"rango de la serie en C : [{temp_c.min():.2f}, {temp_c.max():.2f}]")
print(f"rango de la serie en K : [{temp_k.min():.2f}, {temp_k.max():.2f}]")
print()
c_ant = crps_medio(temp_c, crps_from_quantiles_antiguo)
c_new = crps_medio(temp_c, crps_from_quantiles)
k_ant = crps_medio(temp_k, crps_from_quantiles_antiguo)
k_new = crps_medio(temp_k, crps_from_quantiles)

print(f"{'':<12}{'CRPS medio antiguo':>22}{'CRPS medio corregido':>24}")
print(f"{'Celsius':<12}{c_ant:>22.6f}{c_new:>24.6f}")
print(f"{'Kelvin':<12}{k_ant:>22.6f}{k_new:>24.6f}")
print()
print(f"diferencia C vs K, antiguo   : {abs(c_ant - k_ant):.6f}")
print(f"diferencia C vs K, corregido : {abs(c_new - k_new):.6f}")

rango de la serie en C : [-9.69, 20.46]
rango de la serie en K : [263.46, 293.61]

                CRPS medio antiguo    CRPS medio corregido
Celsius                   5.175068                6.340176
Kelvin                    6.341388                6.340176

diferencia C vs K, antiguo   : 1.166320
diferencia C vs K, corregido : 0.000000


---
## Verificacion de la correccion

El CRPS de una prediccion normal tiene solucion analitica:

$$\mathrm{CRPS}\big(\mathcal{N}(\mu, \sigma), y\big) = \sigma \left[ z\big(2\Phi(z) - 1\big) + 2\phi(z) - \frac{1}{\sqrt{\pi}} \right], \qquad z = \frac{y - \mu}{\sigma}$$

Aproximando esa normal con 99 cuantiles, el valor devuelto debe acercarse al analitico para
cualquier `mu`, incluidos los negativos y los grandes.

In [8]:
def crps_normal_analitico(y, mu, sigma):
    z = (y - mu) / sigma
    return sigma * (z * (2 * norm.cdf(z) - 1) + 2 * norm.pdf(z) - 1 / np.sqrt(np.pi))


q_levels_fino = np.round(np.arange(0.01, 1.0, 0.01), 2)
filas = []
for mu in [-1000, -50, -5, 0, 5, 50, 1000]:
    for desviacion in [0.0, 1.5, 4.0]:
        y = mu + desviacion
        q = norm.ppf(q_levels_fino, loc=mu, scale=2.0)
        filas.append(
            {
                "mu": mu,
                "y - mu": desviacion,
                "analitico": crps_normal_analitico(y, mu, 2.0),
                "antiguo": crps_from_quantiles_antiguo(y, q, q_levels_fino),
                "corregido": crps_from_quantiles(y, q, q_levels_fino),
            }
        )

comp = pd.DataFrame(filas)
comp["err_antiguo"] = (comp["antiguo"] - comp["analitico"]).abs()
comp["err_corregido"] = (comp["corregido"] - comp["analitico"]).abs()
print(comp.round(4).to_string(index=False))
print()
print(f"error maximo antiguo   : {comp['err_antiguo'].max():.4f}")
print(f"error maximo corregido : {comp['err_corregido'].max():.4f}")

   mu  y - mu  analitico  antiguo  corregido  err_antiguo  err_corregido
-1000     0.0     0.4674  -0.4682     0.4674       0.9356         0.0000
-1000     1.5     0.8963  -0.8529     0.8961       1.7492         0.0002
-1000     4.0     2.9056  -2.9930     2.9012       5.8986         0.0043
  -50     0.0     0.4674  -0.1169     0.4674       0.5843         0.0000
  -50     1.5     0.8963  -0.2441     0.8961       1.1404         0.0002
  -50     4.0     2.9056  -0.2441     2.9012       3.1497         0.0043
   -5     0.0     0.4674   0.4670     0.4674       0.0004         0.0000
   -5     1.5     0.8963   0.8938     0.8961       0.0025         0.0002
   -5     4.0     2.9056   2.9033     2.9012       0.0023         0.0043
    0     0.0     0.4674   0.4673     0.4674       0.0001         0.0000
    0     1.5     0.8963   0.8958     0.8961       0.0005         0.0002
    0     4.0     2.9056   2.9007     2.9012       0.0049         0.0043
    5     0.0     0.4674   0.4674     0.4674       

In [9]:
# Propiedades que la version corregida cumple y la antigua no
rng = np.random.default_rng(0)
fallos_antiguo = {"negativo": 0, "no_invariante": 0}
fallos_corregido = {"negativo": 0, "no_invariante": 0}
n_casos = 2000

for _ in range(n_casos):
    k = rng.integers(3, 10)
    lv = np.sort(rng.uniform(0.01, 0.99, k))
    base = np.sort(rng.normal(0, 3, k))
    nivel = rng.uniform(-500, 500)
    q_rand = base + nivel
    y = float(rng.normal(nivel, 5))
    shift = float(rng.uniform(-1000, 1000))

    for funcion, fallos in [
        (crps_from_quantiles_antiguo, fallos_antiguo),
        (crps_from_quantiles, fallos_corregido),
    ]:
        v = funcion(y, q_rand, lv)
        v_shift = funcion(y + shift, q_rand + shift, lv)
        if v < -1e-10:
            fallos["negativo"] += 1
        if not np.isclose(v, v_shift, rtol=1e-6, atol=1e-8):
            fallos["no_invariante"] += 1

print(f"casos evaluados: {n_casos}")
print(f"{'':<12}{'CRPS negativo':>16}{'no invariante':>16}")
print(f"{'antiguo':<12}{fallos_antiguo['negativo']:>16}{fallos_antiguo['no_invariante']:>16}")
print(f"{'corregido':<12}{fallos_corregido['negativo']:>16}{fallos_corregido['no_invariante']:>16}")

casos evaluados: 2000
               CRPS negativo   no invariante
antiguo                  905            2000
corregido                  0               0


---
## La correccion aplicada

```python
xmin = pred_quantiles[0]
xmax = pred_quantiles[-1]

# Fuera del rango de los cuantiles predichos el integrando vale exactamente 0 o 1,
# asi que su contribucion se calcula de forma analitica en lugar de numerica.
tail_area = max(0.0, xmin - y_true) + max(0.0, y_true - xmax)

if xmax > xmin:
    x_values = np.linspace(xmin, xmax, 1000)
    integrand_values = crps_integrand(x_values)
    if hasattr(np, "trapezoid"):
        crps = np.trapezoid(integrand_values, x=x_values)
    else:
        crps = np.trapz(integrand_values, x_values)
    crps = crps + tail_area
else:
    # Todos los cuantiles predichos son iguales, la prediccion es una masa puntual
    crps = tail_area

return float(crps)
```

Tres cambios:

1. La integracion numerica se limita al rango donde el integrando no es constante,
   `[pred_quantiles[0], pred_quantiles[-1]]`. Los 1000 puntos de la rejilla se concentran
   donde hacen falta, asi que la precision ya no depende del nivel de la serie.
2. Las colas se calculan de forma analitica (`tail_area`), que es su valor exacto y no una
   aproximacion truncada.
3. El caso degenerado `xmax == xmin` devuelve `|y_true - q|`, el error absoluto, que es el
   CRPS correcto de una prediccion determinista.

El cambio de `np.__version__ >= "2.0.0"` por `hasattr(np, "trapezoid")` es un extra: la
comparacion de cadenas fallaria con NumPy `"10.0.0"`.

### Tests

En `skforecast/metrics/tests/tests_metrics/test_crps_from_quantiles.py` se ha actualizado el
valor esperado del test existente y se han anadido cuatro tests de regresion:
`test_crps_from_quantiles_is_translation_invariant`,
`test_crps_from_quantiles_is_non_negative`,
`test_crps_from_quantiles_penalizes_y_true_outside_predicted_quantiles` y
`test_crps_from_quantiles_output_when_all_pred_quantiles_are_equal`.